In [ ]:
# standard imports

from pyspark.sql import SparkSession
from pyspark.sql import types as T
from pyspark.sql import functions as F
from pyspark import SparkConf

## Delta Lake vs Apache Hudi Compatibility & Setup Cheat Sheet

<div style="font-size: 0.6em;">

| Component                | Delta Lake                                       | Apache Hudi                                             | Role / Purpose                                                              |
| ------------------------ | ------------------------------------------------ | ------------------------------------------------------- | --------------------------------------------------------------------------- |
| **Python Wrapper**       | `pip install delta-spark`                        | No official PyPI wrapper (uses generic `pyspark`)       | Adds Python-friendly APIs; Delta has better Python ergonomics currently     |
| **Core JAR Package**     | `org.delta-io:delta-core_2.12:2.x`               | `org.apache.hudi:hudi-spark3.3-bundle_2.12:0.14.1`      | Contains the engine (transaction log, ACID, upserts, etc.)                  |
| **Spark SQL Extensions** | `io.delta.sql.DeltaSparkSessionExtension`        | `org.apache.spark.sql.hudi.HoodieSparkSessionExtension` | Enables SQL features like `MERGE`, `UPDATE`, `DELETE`                       |
| **Catalog Registration** | `spark.sql.catalog.spark_catalog = DeltaCatalog` | `spark.sql.catalog.spark_catalog = HoodieCatalog`       | Integrates with Spark's unified catalog (for table registration & metadata) |

</div>

---

### 🔎 Quick Version Discovery

- **Get PySpark Version**
    ```python
    import pyspark
    print(pyspark.__version__)  # e.g., 3.5.5
    ```
- **Check Delta Release**
    - [Delta Releases](https://github.com/delta-io/delta/releases) (find JAR version supporting Spark 3.5.x)
- **Check Scala Version**
    ```python
    spark.sparkContext._gateway.jvm.scala.util.Properties.versionString()
    # Returns something like: 'version 2.12.18'
    ```
- **Find Compatible JAR**
    - Go to [Maven Central - Delta](https://search.maven.org/artifact/io.delta/delta-spark_2.12)
    - Go to [Maven Central - Hudi](https://search.maven.org/artifact/org.apache.hudi/hudi-spark3.5-bundle_2.12)
    - Go to [Maven Central - Iceberg](https://search.maven.org/artifact/org.apache.iceberg/iceberg-spark-runtime-3.5_2.12)
    - Match `spark.jars.packages = '<groupId>:<artifactId>:<version>'` for your Spark/Scala version

---

### ⚙️ Delta Lake Setup Example
```python
from pyspark.sql import SparkSession
from pyspark import SparkConf

conf = SparkConf().setAppName("delta_test")
conf.set('spark.jars.packages', 'io.delta:delta-spark_2.12:3.3.1')
conf.set("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
conf.set("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
conf.set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
spark = SparkSession.builder.config(conf=conf).getOrCreate()
```

---

### ⚙️ Hudi Setup Example
```python
from pyspark.sql import SparkSession
from pyspark import SparkConf

conf = SparkConf().setAppName("hudi_dmltest")
conf.set('spark.jars.packages', 'org.apache.hudi:hudi-spark3.5-bundle_2.12:0.15.0')
conf.set("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
conf.set("spark.sql.extensions", "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
conf.set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.hudi.catalog.HoodieCatalog")
spark = SparkSession.builder.config(conf=conf).getOrCreate()
```

---

### ⚙️ Iceberg Setup Example
```python
from pyspark.sql import SparkSession
from pyspark import SparkConf

conf = SparkConf().setAppName("iceberg_test")
conf.set('spark.jars.packages', 'org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0')
conf.set("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
conf.set("spark.sql.catalog.spark_catalog", "org.apache.iceberg.spark.SparkSessionCatalog")
spark = SparkSession.builder.config(conf=conf).getOrCreate()
```

---

### 📝 Notes
- **ALWAYS match** the Scala version in your JARs with the underlying Spark Scala version (usually 2.12 if installed via pip).
- If using Glue, check AWS Glue docs for supported Spark, Scala, and Delta/Hudi/Iceberg versions.
- Use [Maven Central](https://search.maven.org/) to check for latest compatible bundles.
- Delta Lake has a more mature Python API; Hudi Python API is experimental.
- For Iceberg, Python API support is emerging; most use cases go through PySpark DataFrame API.

---

Feel free to request more code templates, troubleshooting steps, or add advanced scenarios to this cheat sheet!


In [ ]:
conf = SparkConf().setAppName("hudi_dmltest")
conf.set('spark.jars.packages', 'org.apache.hudi:hudi-spark3.5-bundle_2.12:0.15.0')
conf.set("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
conf.set("spark.sql.extensions", "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
conf.set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.hudi.catalog.HoodieCatalog")
spark = SparkSession.builder.config(conf=conf).getOrCreate()

In [ ]:
print (spark.conf.get("spark.sql.warehouse.dir"))

In [ ]:
schema = T.StructType([
    T.StructField('id', T.IntegerType(), False),
    T.StructField('name', T.StringType(), False)
    ])
data = [
    [1, 'Rahul Roy'],
    [2, 'Setu Ghosh'],
    [3, 'Sankadeep Ghosh'],
    [4, 'Rahul Roy']
]

df = spark.createDataFrame(data=data, schema=schema)

In [ ]:
# Define the schema for each table
users_schema = T.StructType([
    T.StructField("user_id", T.StringType(), True),
    T.StructField("name", T.StringType(), True)
])

addresses_schema = T.StructType([
    T.StructField("user_id", T.StringType(), True),
    T.StructField("street", T.StringType(), True),
    T.StructField("city", T.StringType(), True),
    T.StructField("postal_code", T.StringType(), True)
])

favorite_products_schema = T.StructType([
    T.StructField("user_id", T.StringType(), True),
    T.StructField("product_id", T.StringType(), True)
])

preferences_schema = T.StructType([
    T.StructField("user_id", T.StringType(), True),
    T.StructField("key", T.StringType(), True),
    T.StructField("value", T.StringType(), True)
])

extra_info_schema = T.StructType([
    T.StructField("user_id", T.StringType(), True),
    T.StructField("field1", T.StringType(), True),
    T.StructField("field2", T.StringType(), True)
])

additional_info_schema = T.StructType([
    T.StructField("user_id", T.StringType(), True),
    T.StructField("field3", T.StringType(), True),
    T.StructField("field4", T.StringType(), True)
])

# Define data for each table
users_data = [
    ("U123", "John Doe"),
    ("U124", "Jane Doe"),
    ("U125", "Alice Smith"),
    ("U126", "Bob Brown")
]

addresses_data = [
    ("U123", "123 Elm St", "Somewhere", "12345"),
    ("U123", "456 Oak St", "Anywhere", "67890"),
    ("U124", "789 Pine St", "Somewhere", "12345"),
    ("U125", "101 Maple St", "Elsewhere", "54321"),
    ("U126", "202 Birch St", "Nowhere", "98765")
]

favorite_products_data = [
    ("U123", "P001"),
    ("U123", "P002"),
    ("U123", "P003"),
    ("U124", "P004"),
    ("U124", "P005"),
    ("U125", "P006"),
    ("U126", "P007"),
    ("U126", "P008"),
    ("U126", "P009"),
    ("U126", "P010")
]

preferences_data = [
    ("U123", "newsletter", "subscribed"),
    ("U123", "theme", "dark"),
    ("U124", "newsletter", "unsubscribed"),
    ("U124", "theme", "light"),
    ("U124", "language", "en"),
    ("U125", "theme", "dark"),
    ("U126", "language", "fr"),
    ("U126", "currency", "euro"),
    ("U126", "newsletter", "subscribed")
]

extra_info_data = [
    ("U123", "foo", "bar"),
    ("U124", "baz", "qux"),
    ("U125", "abc", "def"),
    ("U126", "ghi", "jkl")
]

additional_info_data = [
    ("U123", "foo1", "bar1"),
    ("U124", "baz1", "qux1"),
    ("U125", "abc1", "def1"),
    ("U126", "ghi1", "jkl1")
]

# Create DataFrames for each table
users_df = spark.createDataFrame(users_data, schema=users_schema)
addresses_df = spark.createDataFrame(addresses_data, schema=addresses_schema)
favorite_products_df = spark.createDataFrame(favorite_products_data, schema=favorite_products_schema)
preferences_df = spark.createDataFrame(preferences_data, schema=preferences_schema)
extra_info_df = spark.createDataFrame(extra_info_data, schema=extra_info_schema)
additional_info_df = spark.createDataFrame(additional_info_data, schema=additional_info_schema)

# Show DataFrames
print("Users Table:")
users_df.show()

print("Addresses Table:")
addresses_df.show()

print("Favorite Products Table:")
favorite_products_df.show()

print("Preferences Table:")
preferences_df.show()

print("Extra Info Table:")
extra_info_df.show()

print("Additional Info Table:")
additional_info_df.show()

In [ ]:
spark.stop()